# BattleActs Actuarial Exam Preparation Products
### DATASCI 530 — Project 1
Maxine Hakimi-Bulson

This project collects publicly available information about six BattleActs
actuarial exam preparation products: Exams 5, 6C, 6U, 7, 8, and 9.
The goal is to construct datasets describing product prices, subscription
durations, advertised features, and public sample-resource links.

Data were collected on October 18, 2026. BattleActs granted written
permission for this one-time educational scraping project and for sharing
the code, dataset, and analysis on GitHub.

Collection was restricted to public product pages and sample-link metadata.
The script did not access accounts, paid course content, or customer data,
and did not download the linked sample files. It paused between products.

## Data Collection Process

Before coding, I manually inspected two products and organized their
information in a spreadsheet to determine the variables to collect.

The algorithm:
1. Opens the BattleActs homepage and the Exams menu.
2. Identifies the six exam-product links.
3. Uses a loop to click each product link and extract its information.
4. Collects sample-card descriptions, button labels, and destination URLs.
5. Converts the extracted records into two pandas DataFrames.

The products dataset contains one row per exam product. The samples
dataset contains one row per advertised sample card, linked to its product
through the product URL.

Feature indicators describe whether a feature was listed in the collected
description and overview sections. "Not listed" does not establish that
the feature is unavailable. Sample resource types are inferred from URL
extensions; file contents and link functionality were not verified.

In [33]:
# Import packages
import pandas as pd
import time
from selenium import webdriver
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Open Chrome and navigate to BattleActs
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

starting_url = "https://battleactsmain.ca/v2/home"
driver.get(starting_url)

click exams

In [2]:
# Open the Exams dropdown
exams_menu = driver.find_element(
    "xpath", '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
)
exams_menu.click()
time.sleep(1)

# Click the Exam 5 product link
exam5_link = driver.find_element(
    "xpath",
    '//a[@class="dropdown-item" and contains(@href, "/product/exam5")]'
)
exam5_link.click()
time.sleep(2)

# Confirm the destination
print(driver.current_url)

https://battleactsmain.ca/v2/product/exam5


extract product name

In [3]:
# Extract the product name
product_name = driver.find_element("xpath", "//h1").text
print("Product:", product_name)

# Display nonempty h2 headings to locate the price
headings = driver.find_elements("xpath", "//h2")

for heading in headings:
    if heading.text.strip():
        print(heading.text)

Product: CAS Exam 5
$500.00 USD
Pass your upcoming exam with confidence.
Our CAS Exam 5 Study Guide
Sample our Study Platform
BattleActs Unique Study Features
Overview Of Our Features
The BattleActs System
Customer Testimonials
More About BattleActs


extract price heading

In [4]:
# Extract the price heading
price_text = driver.find_element(
    "xpath", '//h2[contains(., "$")]'
).text

# Separate the amount and currency
price = float(price_text.split()[0].replace("$", "").replace(",", ""))
currency = price_text.split()[-1]

# Record the source page
product_url = driver.current_url

print("Product:", product_name)
print("Price:", price)
print("Currency:", currency)
print("URL:", product_url)

Product: CAS Exam 5
Price: 500.0
Currency: USD
URL: https://battleactsmain.ca/v2/product/exam5


description

In [5]:
# Locate the study-guide heading
description_heading = driver.find_element(
    "xpath", '//h2[contains(., "Study Guide")]'
)

# Inspect its immediate parent container
description_container = description_heading.find_element("xpath", "..")

print(description_container.get_attribute("outerHTML"))

<div class="col-md-7 col-sm-12 mb-0 font-weight-normal my-5" style="line-height:1.4;">
        <h2 class="color-site mt-3 mb-2 font-weight-bold">Our CAS Exam 5 Study Guide</h2>
        <p class="blue-1 font-weight-bold">A comprehensive study guide with all the study features you expect such as: </p>
        <div class="row">
          <div class="col">
            <ul>
              <li>Comprehensive course manual</li>
              <li>Video Lessons</li>
              <li>Original practice problems in Excel</li>
              <li>All prior CAS exams in Excel</li>
              <li>4 original practice exams in Excel</li>
              <li>Quizzes and flashcards</li>
              <li>Problems in the new item types</li>
            </ul>
          </div>
          <div class="col">
            <ul>
              <li>Discussion forum</li>
              <li>Printable</li>
              <li>Problems with step-by-step solutions</li>
              <li>Study and exam day tips</li>
           

description variable

In [6]:
# Extract the introductory sentence
description_intro = description_container.find_element("xpath", "./p").text

# Extract the bullet points within this container
description_items = description_container.find_elements("xpath", ".//li")
description_bullets = [item.text for item in description_items]

# Combine into one text field
description = description_intro + "\n" + "\n".join(description_bullets)

print(description)

A comprehensive study guide with all the study features you expect such as:
Comprehensive course manual
Video Lessons
Original practice problems in Excel
All prior CAS exams in Excel
4 original practice exams in Excel
Quizzes and flashcards
Problems in the new item types
Discussion forum
Printable
Problems with step-by-step solutions
Study and exam day tips
Formula sheet
Free extension


overview of features

In [7]:
features_heading = driver.find_element(
    "xpath", '//h2[normalize-space()="Overview Of Our Features"]'
)
features_container = features_heading.find_element("xpath", "..")

print(features_container.get_attribute("outerHTML"))

<div class="col-12 col-md-5" style="background-color: #EFEFEF;">
      <h2 class="blue-1 mt-5 mb-3 font-weight-bold text-center">Overview Of Our Features</h2>
      <img src="https://battleactsmain.ca/v2/assets/images/exam5-t.png" class="img-fluid">
    </div>


In [8]:
# Move up to the surrounding layout container
features_section = features_container.find_element("xpath", "..")

print(features_section.get_attribute("outerHTML"))

<div class="row">
    <div class="col-12 col-md-5" style="background-color: #EFEFEF;">
      <h2 class="blue-1 mt-5 mb-3 font-weight-bold text-center">Overview Of Our Features</h2>
      <img src="https://battleactsmain.ca/v2/assets/images/exam5-t.png" class="img-fluid">
    </div>
    <div class="col-12 offset-md-1 col-md-6">
      <div class="row" style="height:16px; width:100%;background-color:#ff6307">
        <div class="col-12"></div>
      </div>
      <ul class="text-dark font-weight-normal mt-4">
        <li>Comprehensive on-line and interactive study manual.</li>
        <li>7-month's access with one free 6-month extension</li>
        <li>Dedicated full-time instructor support in a course forum</li>
        <li>All past CAS exams in Excel 2012 - 2019</li>
        <li>4 original practice exams in Excel</li>
        <li>Hundreds of practice problems in Excel/CBT format</li>
        <li>Step-by-step solutions</li>
        <li>Problems in the new item types</li>
        <li>Conc

extract feature bullet points

In [9]:
# Extract the overview feature list
feature_items = features_section.find_elements("xpath", ".//ul/li")
feature_bullets = [item.text for item in feature_items]

# Combine into one text field
features = "\n".join(feature_bullets)

print(features)
print("\nNumber of feature bullets:", len(feature_bullets))

Comprehensive on-line and interactive study manual.
7-month's access with one free 6-month extension
Dedicated full-time instructor support in a course forum
All past CAS exams in Excel 2012 - 2019
4 original practice exams in Excel
Hundreds of practice problems in Excel/CBT format
Step-by-step solutions
Problems in the new item types
Concise instructional video lessons that supplement the study manual
Interactive online quizzes and flashcards
Quizzes for each sub-section and at the end of a topic
A handy formula and summary sheet
"CHEAT SHEET ON THE GO"
Custom tips for CAS Exam 5 including a study schedule, study tactics and exam day tips
No need to download anything. All access is online.
Print our articles if you wish

Number of feature bullets: 16


In [10]:
import re

# Extract numbers from the advertised features
duration_match = re.search(r"(\d+)-month'?s?\s+access", features, re.I)
extension_match = re.search(r"free\s+(\d+)-month\s+extension", features, re.I)
practice_match = re.search(r"(\d+)\s+original practice exams?", features, re.I)

duration_months = int(duration_match.group(1)) if duration_match else None
free_extension_months = int(extension_match.group(1)) if extension_match else None
practice_exam_count = int(practice_match.group(1)) if practice_match else None

# Check whether features are explicitly listed in the collected sections
listed_text = (description + "\n" + features).lower()

video_lessons_listed = "Yes" if "video lessons" in listed_text else "Not listed"
formula_sheet_listed = (
    "Yes" if "formula sheet" in listed_text
    or "formula and summary sheet" in listed_text else "Not listed"
)
cheat_sheet_listed = (
    "Yes" if "cheat sheet on the go" in listed_text else "Not listed"
)

# Display the extracted values
display(pd.DataFrame([{
    "duration_months": duration_months,
    "free_extension_months": free_extension_months,
    "practice_exam_count": practice_exam_count,
    "video_lessons_listed": video_lessons_listed,
    "formula_sheet_listed": formula_sheet_listed,
    "cheat_sheet_listed": cheat_sheet_listed
}]))

,duration_months,free_extension_months,practice_exam_count,video_lessons_listed,formula_sheet_listed,cheat_sheet_listed
0,7,6,4,Yes,Yes,Yes


unique study features

In [11]:
# Find the section containing the comparison table
study_heading = driver.find_element(
    "xpath", '//h2[normalize-space()="BattleActs Unique Study Features"]'
)
study_container = study_heading.find_element("xpath", "..")

print(study_container.get_attribute("outerHTML"))

<div class="col-12 text-center">
        <h2 class="color-site mt-4 mb-3 font-weight-bold">BattleActs Unique Study Features</h2>
        <p class="blue-1 font-weight-bold">BattleActs users have access to extra features not offered by other study guide providers.</p>
      </div>


In [12]:
study_section = study_container.find_element("xpath", "..")

print(study_section.get_attribute("outerHTML"))

<div class="row">
      <div class="col-12 text-center">
        <h2 class="color-site mt-4 mb-3 font-weight-bold">BattleActs Unique Study Features</h2>
        <p class="blue-1 font-weight-bold">BattleActs users have access to extra features not offered by other study guide providers.</p>
      </div>
    </div>


In [13]:
# Locate the comparison table following the heading
study_table = study_heading.find_element("xpath", "following::table[1]")

print(study_table.get_attribute("outerHTML"))

<table class="table table-bordered font-weight-normal" style="line-height:1.2;">
      <thead>
        <tr>
          <th class="text-center bg-orange-1">Study Feature</th>
          <th class="text-center bg-blue-2">BattleActs</th>
          <th class="text-center bg-gray-1 text-nowrap">Other Guides</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td class="bg-orange-2">Interactive study manual</td>
          <td class="bg-blue-3">Stay engaged with our interactive features such as videos and embedded quizzes</td>
          <td class="text-center bg-gray-2 font-weight-bolder">X</td>
        </tr>
        <tr>
          <td class="bg-orange-2">Web-based practice problems</td>
          <td class="bg-blue-3">Generate an infinite number of randomized practice problems that can be solved in Excel</td>
          <td class="text-center bg-gray-2 font-weight-bolder">X</td>
        </tr>
        <tr>
          <td class="bg-orange-2">Interactive exam readiness tracker</t

In [14]:
# Extract feature names and BattleActs descriptions
study_rows = study_table.find_elements("xpath", "./tbody/tr")

study_feature_details = []

for row in study_rows:
    cells = row.find_elements("xpath", "./td")
    feature_name = cells[0].text.strip()
    feature_description = cells[1].text.strip()

    study_feature_details.append(
        feature_name + ": " + feature_description
    )

# Combine into one field for the product dataset
study_features = "\n".join(study_feature_details)

print(study_features)
print("\nNumber of study features:", len(study_feature_details))

Interactive study manual: Stay engaged with our interactive features such as videos and embedded quizzes
Web-based practice problems: Generate an infinite number of randomized practice problems that can be solved in Excel
Interactive exam readiness tracker: Answering quizzes correctly boosts your exam readiness score (0-100%).
Quiz filters: Easily select and target where you need more practice
Guided study schedule: Create an efficient study schedule with tips on how much time to spend on the material
Catalog of most frequently asked exam questions: Find top questions quickly and easily with this feature

Number of study features: 6


introductory paragraph

In [15]:
# Find the introductory heading and the first paragraph after it
intro_heading = driver.find_element(
    "xpath",
    '//h2[normalize-space()="Pass your upcoming exam with confidence."]'
)

intro_text = intro_heading.find_element(
    "xpath", "following-sibling::p[1]"
).text

print(intro_text)

Gear up for your first upper level CAS exam with BattleActs. Get organized quickly with our study schedule, master the content with our online course, practice with our hundreds of problems, and use our quiz progress tracker.


FAQ: how this exam differs

In [16]:
# Locate the exam-differences question
differences_question = driver.find_element(
    "xpath",
    '//*[contains(normalize-space(.), "different than other")'
    ' and not(.//*[contains(normalize-space(.), "different than other")])]'
)

print(differences_question.get_attribute("outerHTML"))
print(differences_question.find_element("xpath", "..").get_attribute("outerHTML"))

<p class="text-dark">How is CAS Exam 5 different than other actuarial exams?</p>
<div class="container product-section">
    <h2 class="color-site my-5 text-center font-weight-bold">More About BattleActs</h2>
    <img src="https://battleactsmain.ca/v2/assets/images/computer.png" class="float-right ml-3 mb-3 img-fluid" style="width:500px;" title="Battleacts Testimonials">
    <p class="text-justify">
      </p><p class="text-dark">How long will the BattleActs platform be available to me?</p>
      <p class="font-weight-normal">Your membership is available for a period of 7 months. If you do not sit for or pass your exam, we will give you one free 6-month extension for any future sitting. Please email us at <a href="mailto:info@battleacts.ca">info@battleacts.ca</a> in order to request this extension. </p>
      <p class="text-dark">Do you offer a discount if I have used my free extension?</p>
      <p class="font-weight-normal">Yes. Subscribers who have already used their free extension 

In [17]:
# Extract the answer immediately following the FAQ question
exam_differences = differences_question.find_element(
    "xpath", "following-sibling::p[1]"
).text.strip()

print(exam_differences)

CAS Exam 5 requires a deep understanding of the concepts and to be able to apply these concepts quickly on exam day. The way to reach this mastery is to thoroughly learn the material and to practice, practice, practice. We are the only study guide that provides an infinite number of calculational problems as our web-based platform randomly generates numbers for optimal learning.


First Row

In [18]:
# Record when this product's data were collected
collection_date = pd.Timestamp.now(tz="America/New_York").isoformat(
    timespec="seconds"
)

# Assemble the extracted information
product_data = {
    "product_name": product_name,
    "price": price,
    "currency": currency,
    "description": description,
    "features": features,
    "duration_months": duration_months,
    "product_url": product_url,
    "collection_date": collection_date,
    "study_features": study_features,
    "exam_differences": exam_differences,
    "intro_text": intro_text,
    "practice_exam_count": practice_exam_count,
    "free_extension_months": free_extension_months,
    "video_lessons_listed": video_lessons_listed,
    "formula_sheet_listed": formula_sheet_listed,
    "cheat_sheet_listed": cheat_sheet_listed
}

# Display vertically so every field is easy to review
product_df = pd.DataFrame([product_data])
display(product_df.T)

,0
product_name,CAS Exam 5
price,500.0
currency,USD
description,A comprehensive study guide with all the study...
features,Comprehensive on-line and interactive study ma...
duration_months,7
product_url,https://battleactsmain.ca/v2/product/exam5
collection_date,2026-10-18T22:27:41-04:00
study_features,Interactive study manual: Stay engaged with ou...
exam_differences,CAS Exam 5 requires a deep understanding of th...


sample our study platform section

In [19]:
# Locate the sample section heading
sample_heading = driver.find_element(
    "xpath", '//h2[normalize-space()="Sample our Study Platform"]'
)

# Inspect the surrounding container
sample_container = sample_heading.find_element("xpath", "..")

print(sample_container.get_attribute("outerHTML"))

<div class="col-12 text-center">
      <h2 class="color-site mt-5 mb-3 font-weight-bold">Sample our Study Platform</h2>
      <p class="blue-1 font-weight-bold">Explore our resources and study materials below:</p>
    </div>


In [20]:
sample_section = sample_container.find_element("xpath", "..")

print(sample_section.get_attribute("outerHTML"))

<div class="row">
    <div class="col-12 text-center">
      <h2 class="color-site mt-5 mb-3 font-weight-bold">Sample our Study Platform</h2>
      <p class="blue-1 font-weight-bold">Explore our resources and study materials below:</p>
    </div>
  </div>


In [21]:
sample_section = sample_section.find_element("xpath", "..")

# Print in full, avoiding notebook output truncation
sample_html = sample_section.get_attribute("outerHTML")
print(sample_html)

<div class="container product-section">
  <div class="row">
    <div class="col-12 text-center">
      <h2 class="color-site mt-5 mb-3 font-weight-bold">Sample our Study Platform</h2>
      <p class="blue-1 font-weight-bold">Explore our resources and study materials below:</p>
    </div>
  </div>
  <div class="owl-carousel owl-theme pt-4 owl-loaded owl-drag">
    
    
    
    
    
        
  <div class="owl-stage-outer"><div class="owl-stage" style="transform: translate3d(0px, 0px, 0px); transition: all; width: 1110px;"><div class="owl-item active" style="width: 185px;"><div>
      <div class="text-center px-4">
        <div class="feature-icon ml-auto mb-4"><img src="https://battleactsmain.ca/v2/assets/images/5.png"></div>
        <p>Comprehensive on-line and interactive study manual</p>
        <a href="https://battleactsmain.ca/v2/assets/doc/Exam_5_sneak_peek_Werner_11.pdf" target="_blank" class="btn btn-sm btn-white btn-primary btn-primary-light rounded-pill">SEE SAMPLE</a>
    

In [22]:
# Find sample cards, excluding duplicate carousel clones
sample_cards = sample_section.find_elements(
    "xpath",
    './/div[contains(concat(" ", normalize-space(@class), " "), " owl-item ")'
    ' and not(contains(concat(" ", normalize-space(@class), " "), " cloned "))]'
)

sample_data = []

for card in sample_cards:
    # Read textContent so off-screen carousel cards are included
    sample_label = card.find_element("xpath", ".//p").get_attribute(
        "textContent"
    ).strip()

    control = card.find_element("xpath", ".//a[@href] | .//button[@data-url]")
    sample_url = (
        control.get_attribute("href")
        or control.get_attribute("data-url")
    )
    button_text = control.get_attribute("textContent").strip()

    # Infer the resource type from the URL, without opening the file
    url_path = sample_url.split("?")[0].split("#")[0].lower()

    if url_path.endswith(".pdf"):
        resource_type = "PDF"
    elif url_path.endswith((".xlsx", ".xls")):
        resource_type = "Excel"
    elif url_path.endswith(".mp4"):
        resource_type = "Video"
    else:
        resource_type = "Unknown"

    sample_data.append({
        "product_name": product_name,
        "sample_label": sample_label,
        "button_text": button_text,
        "sample_url": sample_url,
        "resource_type": resource_type,
        "product_url": product_url,
        "collection_date": collection_date
    })

samples_df = pd.DataFrame(sample_data)
display(samples_df)

,product_name,sample_label,button_text,sample_url,resource_type,product_url,collection_date
0,CAS Exam 5,Comprehensive on-line and interactive study ma...,SEE SAMPLE,https://battleactsmain.ca/v2/assets/doc/Exam_5...,PDF,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00
1,CAS Exam 5,Handy formula and summary sheet,SEE SAMPLE,https://battleactsmain.ca/v2/assets/doc/Cheat_...,PDF,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00
2,CAS Exam 5,Past CAS Exams in Excel,SEE SAMPLE,https://battleactsmain.ca/v2/assets/doc/Cheat_...,PDF,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00
3,CAS Exam 5,Concise instructional video lessons that suppl...,VIDEO,https://battleactsmain.ca/v2/assets/videos/vid...,Video,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00
4,CAS Exam 5,Excel practice problems and 4 original practic...,DOWNLOAD,https://www.battleacts5.ca/Excel/BA_Exams/BA_E...,Excel,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00
5,CAS Exam 5,Interactive online quizzes and flashcards,VIDEO,https://battleactsmain.ca/v2/assets/videos/BA_...,Video,https://battleactsmain.ca/v2/product/exam5,2026-10-18T22:27:41-04:00


In [23]:
# Check sample counts and shared destinations
print("Sample cards:", len(samples_df))
print("Unique sample URLs:", samples_df["sample_url"].nunique())

display(samples_df["resource_type"].value_counts())

Sample cards: 6
Unique sample URLs: 5


resource_type
PDF      3
Video    2
Excel    1
Name: count, dtype: int64

Extraction Function

In [24]:
def extract_product(driver):
    # Return None when an optional text field is absent
    def get_text(xpath):
        elements = driver.find_elements("xpath", xpath)
        return elements[0].text.strip() if elements else None

    # Basic product information
    product_name = get_text("//h1")
    price_text = get_text('//h2[contains(., "$")]')

    price = (
        float(price_text.split()[0].replace("$", "").replace(",", ""))
        if price_text else None
    )
    currency = price_text.split()[-1] if price_text else None

    # Description and its bullet points
    description_parts = driver.find_elements(
        "xpath",
        '//h2[contains(., "Study Guide")]/parent::div/p'
        ' | //h2[contains(., "Study Guide")]/parent::div//li'
    )
    description = "\n".join(
        element.text.strip() for element in description_parts
    ) or None

    # Overview feature bullets
    feature_items = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Overview Of Our Features"]'
        '/parent::div/parent::div//ul/li'
    )
    features = "\n".join(
        item.text.strip() for item in feature_items
    ) or None

    # Structured fields derived from the collected text
    feature_text = features or ""
    duration_match = re.search(
        r"(\d+)-month'?s?\s+access", feature_text, re.I
    )
    extension_match = re.search(
        r"free\s+(\d+)-month\s+extension", feature_text, re.I
    )
    practice_match = re.search(
        r"(\d+)\s+original practice exams?", feature_text, re.I
    )

    listed_text = ((description or "") + "\n" + feature_text).lower()

    # Comparison table: retain only feature names and BA descriptions
    study_details = []
    study_rows = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="BattleActs Unique Study Features"]'
        '/following::table[1]/tbody/tr'
    )

    for row in study_rows:
        cells = row.find_elements("xpath", "./td")
        if len(cells) >= 2:
            study_details.append(
                cells[0].text.strip() + ": " + cells[1].text.strip()
            )

    # Introductory text and exam-specific FAQ
    intro_text = get_text(
        '//h2[normalize-space()="Pass your upcoming exam with confidence."]'
        '/following-sibling::p[1]'
    )
    exam_differences = get_text(
        '//p[contains(., "different than other")]'
        '/following-sibling::p[1]'
    )

    product_url = driver.current_url
    collection_date = pd.Timestamp.now(
        tz="America/New_York"
    ).isoformat(timespec="seconds")

    product = {
        "product_name": product_name,
        "price": price,
        "currency": currency,
        "description": description,
        "features": features,
        "duration_months": (
            int(duration_match.group(1)) if duration_match else None
        ),
        "product_url": product_url,
        "collection_date": collection_date,
        "study_features": "\n".join(study_details) or None,
        "exam_differences": exam_differences,
        "intro_text": intro_text,
        "practice_exam_count": (
            int(practice_match.group(1)) if practice_match else None
        ),
        "free_extension_months": (
            int(extension_match.group(1)) if extension_match else None
        ),
        "video_lessons_listed": (
            "Yes" if "video lessons" in listed_text else "Not listed"
        ),
        "formula_sheet_listed": (
            "Yes" if "formula sheet" in listed_text
            or "formula and summary sheet" in listed_text else "Not listed"
        ),
        "cheat_sheet_listed": (
            "Yes" if "cheat sheet on the go" in listed_text else "Not listed"
        )
    }

    # Sample cards within the section containing the sample heading
    sample_cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        '/ancestor::div[contains(concat(" ", normalize-space(@class), " "),'
        ' " product-section ")][1]'
        '//div[contains(concat(" ", normalize-space(@class), " "),'
        ' " owl-item ")'
        ' and not(contains(concat(" ", normalize-space(@class), " "),'
        ' " cloned "))]'
    )

    samples = []

    for card in sample_cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements(
            "xpath", ".//a[@href] | .//button[@data-url]"
        )

        if not labels or not controls:
            raise ValueError(
                "Unexpected sample-card structure on " + product_url
            )

        control = controls[0]
        sample_url = (
            control.get_attribute("href")
            or control.get_attribute("data-url")
        )
        url_path = (sample_url or "").split("?")[0].split("#")[0].lower()

        if url_path.endswith(".pdf"):
            resource_type = "PDF"
        elif url_path.endswith((".xlsx", ".xls")):
            resource_type = "Excel"
        elif url_path.endswith(".mp4"):
            resource_type = "Video"
        else:
            resource_type = "Unknown"

        samples.append({
            "product_name": product_name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": control.get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": resource_type,
            "product_url": product_url,
            "collection_date": collection_date
        })

    return product, samples

Click through all 6 exams

In [25]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 15)

# Open the homepage and collect the exam-link destinations
driver.get(starting_url)

menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
links_xpath = (
    '//a[contains(@class, "dropdown-item")'
    ' and contains(@href, "/v2/product/")]'
)

wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

exam_links = driver.find_elements("xpath", links_xpath)

# Remove repeated URLs while preserving their order
exam_urls = list(dict.fromkeys(
    link.get_attribute("href") for link in exam_links
))

if len(exam_urls) != 6:
    raise ValueError(
        f"Expected 6 exam links, but found {len(exam_urls)}. Check the menu."
    )

products = []
all_samples = []

for exam_url in exam_urls:
    # Reopen the homepage and menu for each product
    driver.get(starting_url)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

    # Navigate by actually clicking the product link
    product_link_xpath = (
        '//a[contains(@class, "dropdown-item")'
        f' and @href="{exam_url}"]'
    )
    wait.until(
        EC.element_to_be_clickable((By.XPATH, product_link_xpath))
    ).click()

    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # Allow the sample carousel to initialize

    product, samples = extract_product(driver)

    # Preserve completed rows as the loop progresses
    products.append(product)
    all_samples.extend(samples)

    print(
        f'{product["product_name"]}: '
        f'{product["price"]} {product["currency"]}; '
        f'{len(samples)} sample cards'
    )

    time.sleep(3)  # Pause between products

# Build the two datasets
products_df = pd.DataFrame(products)
samples_df = pd.DataFrame(all_samples)

display(products_df[
    ["product_name", "price", "currency", "duration_months"]
])
print("Total sample cards:", len(samples_df))

CAS Exam 5: 500.0 USD; 6 sample cards
CAS Exam 6C: 1000.0 CAD; 3 sample cards
CAS Exam 6U: 1000.0 USD; 6 sample cards
CAS Exam 7: 1000.0 USD; 4 sample cards
CAS Exam 8: 1000.0 USD; 3 sample cards
CAS Exam 9: 1000.0 USD; 3 sample cards


,product_name,price,currency,duration_months
0,CAS Exam 5,500.0,USD,7
1,CAS Exam 6C,1000.0,CAD,7
2,CAS Exam 6U,1000.0,USD,7
3,CAS Exam 7,1000.0,USD,7
4,CAS Exam 8,1000.0,USD,7
5,CAS Exam 9,1000.0,USD,7


Total sample cards: 25


look at collected data

In [26]:
# Dataset sizes and duplicate checks
print("Products:", len(products_df))
print("Sample cards:", len(samples_df))
print("Duplicate product URLs:", products_df["product_url"].duplicated().sum())
print("Duplicate sample rows:", samples_df.duplicated().sum())

# Missing values, including empty text
display(
    products_df.replace(r"^\s*$", pd.NA, regex=True)
    .isna().sum().to_frame("Missing values")
)
display(
    samples_df.replace(r"^\s*$", pd.NA, regex=True)
    .isna().sum().to_frame("Missing values")
)

# Review the structured product fields
display(products_df[[
    "product_name", "practice_exam_count", "free_extension_months",
    "video_lessons_listed", "formula_sheet_listed", "cheat_sheet_listed"
]])

# Inspect sample cards without a usable URL
missing_url = (
    samples_df["sample_url"].fillna("").str.strip().eq("")
)

display(samples_df.loc[
    missing_url,
    ["product_name", "sample_label", "button_text", "sample_url"]
])

# Save the current datasets
products_df.to_csv("battleacts_products.csv", index=False)
samples_df.to_csv("battleacts_samples.csv", index=False)

Products: 6
Sample cards: 25
Duplicate product URLs: 0
Duplicate sample rows: 0


,Missing values
product_name,0
price,0
currency,0
description,0
features,0
duration_months,0
product_url,0
collection_date,0
study_features,0
exam_differences,0


,Missing values
product_name,0
sample_label,0
button_text,0
sample_url,1
resource_type,0
product_url,0
collection_date,0


,product_name,practice_exam_count,free_extension_months,video_lessons_listed,formula_sheet_listed,cheat_sheet_listed
0,CAS Exam 5,4,6,Yes,Yes,Yes
1,CAS Exam 6C,1,6,Not listed,Not listed,Not listed
2,CAS Exam 6U,2,6,Yes,Yes,Yes
3,CAS Exam 7,2,6,Not listed,Not listed,Not listed
4,CAS Exam 8,2,6,Not listed,Not listed,Not listed
5,CAS Exam 9,2,6,Not listed,Not listed,Not listed


,product_name,sample_label,button_text,sample_url
24,CAS Exam 9,Excel practice problems and 2 original practic...,Coming Soon,None


sample cards

In [27]:
# Classify whether each sample card has a destination
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")
coming_soon = samples_df["button_text"].str.contains(
    "coming soon", case=False, na=False
)

samples_df["sample_status"] = "No URL listed"
samples_df.loc[has_url, "sample_status"] = "Link listed"
samples_df.loc[~has_url & coming_soon, "sample_status"] = "Coming soon"

# Summarize sample cards by product and status
sample_summary = pd.crosstab(
    samples_df["product_name"],
    samples_df["sample_status"]
)

display(sample_summary)

# Update the saved samples dataset
samples_df.to_csv("battleacts_samples.csv", index=False)

sample_status,Coming soon,Link listed
product_name,,
CAS Exam 5,0,6
CAS Exam 6C,0,3
CAS Exam 6U,0,6
CAS Exam 7,0,4
CAS Exam 8,0,3
CAS Exam 9,1,2


In [28]:
numeric_summary = products_df.groupby("currency")[
    ["price", "duration_months", "practice_exam_count", "free_extension_months"]
].agg(["count", "mean", "var"])

display(numeric_summary.round(2))

price                  duration_months           practice_exam_count  \
         count    mean      var           count mean  var               count   
currency                                                                        
CAD          1  1000.0      NaN               1  7.0  NaN                   1   
USD          5   900.0  50000.0               5  7.0  0.0                   5   

                   free_extension_months            
         mean  var                 count mean  var  
currency                                            
CAD       1.0  NaN                     1  6.0  NaN  
USD       2.4  0.8                     5  6.0  0.0

In [29]:
# Price statistics: keep currencies separate
price_summary = (
    products_df.groupby("currency")["price"]
    .agg(["count", "mean", "var"])
    .rename(columns={
        "count": "Products",
        "mean": "Mean price",
        "var": "Price variance"
    })
)

display(price_summary.round(2))

# Other numeric statistics: include all six products
numeric_summary = (
    products_df[
        ["duration_months", "practice_exam_count", "free_extension_months"]
    ]
    .agg(["count", "mean", "var"])
    .T
    .rename(index={
        "duration_months": "Standard access (months)",
        "practice_exam_count": "Original practice exams",
        "free_extension_months": "Conditional free extension (months)"
    }, columns={
        "count": "Observations",
        "mean": "Mean",
        "var": "Variance"
    })
)

display(numeric_summary.round(2))

,Products,Mean price,Price variance
currency,,,
CAD,1,1000.0,NaN
USD,5,900.0,50000.0


,Observations,Mean,Variance
Standard access (months),6.0,7.00,0.00
Original practice exams,6.0,2.17,0.97
Conditional free extension (months),6.0,6.00,0.00


The dataset contains six exam products. Five are priced in USD, averaging $900, and one is priced at CAD 1,000. All products advertise seven months of standard access and a conditional six-month free extension. The sample-resource dataset contains 25 cards: 24 have destination URLs, while one Exam 9 resource is marked “Coming Soon.” The CAD price variance is undefined because there is only one CAD observation.

## Data Quality

The collection produced six product observations and 25 sample-card
observations. No product URLs or complete sample rows were duplicated.
All product fields were populated.

One sample URL was missing. Inspection showed that the associated Exam 9
card was labeled "Coming Soon"; this record was retained and classified
accordingly. Shared destination URLs were also retained because separate
sample cards can advertise different resources using the same link.

Exam 5 results were compared with the manually collected values:
USD 500, seven months of access, a conditional six-month extension,
and four original practice exams.

Prices were stored numerically, with currency in a separate column.
USD and CAD prices were not pooled when calculating price statistics.

## Discussion

The five USD-priced products have an average listed price of $900.
Exam 6C is priced separately at CAD 1,000. All six products advertise
seven months of standard access and a conditional six-month free extension.

The public sample section contains 25 advertised cards, of which 24 have
destination URLs. One Exam 9 card is marked "Coming Soon." Some cards
share a destination, so card counts do not represent counts of unique files.

These data describe one provider's public offerings at one point in time.
They do not measure course quality, exam outcomes, or independently
establish differences between BattleActs and its competitors.

In [ ]:
# Save both datasets
products_df.to_csv("battleacts_products.csv", index=False)
samples_df.to_csv("battleacts_samples.csv", index=False)

In [ ]:
driver.quit()